In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import jax
from jax.typing import ArrayLike
from jax import numpy as jnp # JAX version of NumPy.
from flax import nnx # Neural networks in JAX.
import optax # Optimization in JAX.
import gymnasium as gym
import numpy as np # For compatibility with gymnasium.
from tqdm import tqdm
import pennylane as qml
import json
import yaml
import tyro
from pathlib import Path

In [3]:
import qnrl

In [4]:
jax.config.update("jax_enable_x64", True) # Enable double-precision (default is single-precision).

In [5]:
jax.devices()

[CpuDevice(id=0)]

In [6]:
jax.default_backend()

'cpu'

In [7]:
# nnx.Linear

# TODO what if the `QuantumNativeQNetwork` module takes as input a probability distribution over policies from another source. 

# TODO linear layer that encodes input observations into circuit shape.
# TODO separate encoder+parameterized_unitary for every action.


In [8]:
batch_size = 128
num_actions = 5
# num_actions = 3
din = 4

In [9]:
model = qnrl.modules.QuantumNativeQNetwork(
    # diff_method="backprop",
    # diff_method=None,
    shots=100,
    # shots=None,
    din=din,
    n_actions = num_actions,
    n_returns=2**7, # 101
    # n_returns=2**3, # 101
    z_min=-100,
    z_max=100,
    m_moment=1,
    n_reuploads=5,
    # n_reuploads=1,
    n_layers=5,
    rngs_kwargs=dict(default=0),
)
nnx.display(model)
# nnx.display(model.returns_probs_matrix.sum(axis=-1))

In [10]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)
observations = jax.random.uniform(subkey, shape=(1,din,)) # Single observation for circuit drawing.

print(qml.draw(model.circuit)(
    inputs=observations,
    weights=model.circ_weights,
    n_actions=model.n_actions,
    d_actions=model.d_actions,
    d_returns=model.d_returns,
    m_moment=model.m_moment,
    n_reuploads=model.n_reuploads,
    n_layers=model.n_layers,
    U_atoms=model.U_atoms,
))

0: ─╭|Ψ⟩ ···
1: ─├|Ψ⟩ ···
2: ─╰|Ψ⟩ ···
3: ───── ···
4: ───── ···
5: ───── ···
6: ───── ···
7: ───── ···
8: ───── ···
9: ───── ···

0: ··· ─╭Select(1.10,0.42,0.57,1.10,0.42,0.57,1.10,0.42,0.57,1.10,0.42,0.57,1.97,0.38,2.09,0.49,0.60,2.21,0.18,1.01,1.10,0.08,0.08,0.08,0.08,0.08,0.08,0.08,1.10,0.42,0.57,1.10,0.42,0.57,1.10,0.42,0.57,1.10,0.42,0.57,1.97,0.38,2.09,0.49,0.60,2.21,0.18,1.01,1.10,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.82,2.68,0.78,0.82,2.68,0.78,0.82,2.68,0.78,0.82,2.68,0.78,0.40,2.83,1.83,2.25,2.76,2.83,1.59,2.44,2.74,0.08,0.08,0.08,0.08,0.08,0.08,0.08,2.68,1.96,0.86,2.68,1.96,0.86,2.68,1.96,0.86,2.68,1.96,0.86,1.84,2.46,0.03,1.44,1.28,1.54,0.25,0.74,3.01,0.08,0.08,0.08,0.08,0.08,0.08,0.08,1.82,1.88,2.01,1.82,1.88,2.01,1.82,1.88,2.01,1.82,1.88,2.01,1.86,0.33,1.00,1.35,2.64,2.40,1.13,1.15,1.19,0.08,0.08,0.08,0.08,0.08,0.08,0.08,0.16,2.91,2.40,0.16,2.91,2.40,0.16,2.91,2.40,0.16,2.91,2.40,1.10,2.61,2.42,1.17,0.56,2.49,2.64,3.00,1.93,0.11,1.50,1.74,0.11,1.50,1.74,0.11,1.50,1.74,0.1

In [11]:
# Pre-compile a single observation variant.
key, subkey = jax.random.split(key)
model(jax.random.uniform(subkey, shape=(1,din,))).argmax(axis=-1)

Array([4], dtype=int64)

In [12]:
# Generate a batched observation.
key, subkey = jax.random.split(key)
observations = jax.random.uniform(subkey, shape=(batch_size,din,))

In [13]:
probs_actions = model(observations)

In [14]:
best_actions = probs_actions.argmax(axis=-1)
best_actions
# nnx.display(best_actions)

Array([4, 3, 3, 3, 3, 4, 3, 3, 2, 4, 3, 4, 4, 3, 4, 4, 4, 3, 4, 3, 3, 2,
       4, 3, 3, 4, 4, 3, 4, 3, 3, 1, 3, 3, 0, 4, 3, 3, 3, 3, 4, 3, 3, 0,
       3, 3, 0, 0, 4, 3, 3, 4, 4, 3, 3, 4, 3, 4, 0, 4, 3, 3, 3, 3, 4, 3,
       3, 3, 3, 4, 3, 4, 3, 3, 3, 3, 3, 2, 0, 3, 3, 3, 3, 3, 4, 3, 4, 3,
       0, 3, 4, 3, 0, 4, 4, 3, 3, 3, 4, 4, 3, 4, 3, 3, 0, 3, 3, 3, 3, 0,
       4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 4], dtype=int64)

In [15]:
nnx.display(probs_actions.T)

In [16]:
nnx.display(probs_actions[:, jnp.array(model.active_action_idx, dtype=int)].T)

In [17]:
nnx.display(nnx.softmax(probs_actions[:, jnp.array(model.active_action_idx, dtype=int)], axis=-1).T)